# 09 — Commercial Density

Computes shop-specific density and diversity metrics per census tract from OSM.

**Data source:** Overpass API (OSM `shop=*`) — fully portable.

**Method:** Single batch Overpass query for all shops in the study area, then BallTree matching to assign each shop to its nearest tract.

**Distinct from notebook 02:** This focuses exclusively on shops (retail/commercial), computing shop-specific metrics like type entropy and brand ratio. Notebook 02 covers all amenity categories broadly.

**Output columns:** `tract_id`, `shop_count`, `shop_density_km2`, `shop_type_entropy`, `brand_ratio`

**Output file:** `csv/09_commercial_density.csv`

In [ ]:
ZONES_CONFIG = "zones.json"
QUERY_RADIUS = 500

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import json
import os
import math
import hashlib
from collections import Counter
from sklearn.neighbors import BallTree

os.makedirs("csv", exist_ok=True)
os.makedirs("cache", exist_ok=True)

df_tracts = pd.read_csv("csv/01_zone_definition.csv", dtype={"tract_id": str})
print(f"Loaded {len(df_tracts)} tracts")

# Bounding box
BUFFER = 0.015
LAT_MIN = df_tracts["tract_lat"].min() - BUFFER
LAT_MAX = df_tracts["tract_lat"].max() + BUFFER
LON_MIN = df_tracts["tract_lon"].min() - BUFFER
LON_MAX = df_tracts["tract_lon"].max() + BUFFER
BBOX = f"{LAT_MIN},{LON_MIN},{LAT_MAX},{LON_MAX}"

In [ ]:
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "zone-finding/1.0 (research project)"}


def _cache_path(query):
    h = hashlib.sha1(query.encode()).hexdigest()
    return f"cache/{h}.json"


def query_overpass_cached(query, max_retries=3):
    cp = _cache_path(query)
    if os.path.exists(cp):
        with open(cp, encoding="utf-8") as f:
            return json.load(f)
    last_error = None
    for attempt in range(max_retries):
        ep = OVERPASS_ENDPOINTS[attempt % len(OVERPASS_ENDPOINTS)]
        try:
            r = requests.post(ep, data={"data": query}, headers=HEADERS, timeout=90)
            r.raise_for_status()
            data = r.json()
            with open(cp, "w", encoding="utf-8") as f:
                json.dump(data, f)
            return data
        except Exception as e:
            last_error = e
            time.sleep(3 + attempt * 2)
    raise RuntimeError(f"Overpass failed: {last_error}")


def shannon_entropy(counts):
    """Shannon entropy from a Counter of type → count."""
    total = sum(counts.values())
    if total == 0:
        return 0.0
    proportions = np.array([c / total for c in counts.values()])
    proportions = proportions[proportions > 0]
    return -np.sum(proportions * np.log2(proportions))


print("Helpers ready.")

In [ ]:
# ── Batch query: ALL shops in study area ──────────────

query = (f'[out:json][timeout:90];\n'
         f'(node["shop"]({BBOX});\n'
         f' way["shop"]({BBOX}););\n'
         f'out center tags;')

print("Querying all shops...")
data = query_overpass_cached(query)

# Extract shop records with coordinates
EARTH_RADIUS_M = 6371000
AREA_KM2 = math.pi * (QUERY_RADIUS / 1000) ** 2
MAX_DIST_M = QUERY_RADIUS

poi_records = []
for el in data.get("elements", []):
    tags = el.get("tags", {})
    shop_val = tags.get("shop", "")
    if shop_val and shop_val not in {"vacant", "yes"}:
        lat = el.get("lat") or (el.get("center", {}) or {}).get("lat")
        lon = el.get("lon") or (el.get("center", {}) or {}).get("lon")
        if lat and lon:
            poi_records.append({
                "lat": float(lat), "lon": float(lon),
                "shop_type": shop_val,
                "has_brand": bool(tags.get("brand")),
            })

print(f"  Found {len(poi_records)} shops")

# ── BallTree: assign each shop to nearest tract ───────

tract_coords_rad = np.radians(df_tracts[["tract_lat", "tract_lon"]].values)
tract_ids = df_tracts["tract_id"].tolist()

# Per-tract accumulators
tract_shop_types = {t: Counter() for t in tract_ids}
tract_brand_count = {t: 0 for t in tract_ids}

if poi_records:
    tract_tree = BallTree(tract_coords_rad, metric="haversine")
    poi_coords = np.radians([[p["lat"], p["lon"]] for p in poi_records])
    distances, indices = tract_tree.query(poi_coords, k=1)

    for j, (dist, idx) in enumerate(zip(distances.flatten(), indices.flatten())):
        if dist * EARTH_RADIUS_M <= MAX_DIST_M:
            tid = tract_ids[idx]
            tract_shop_types[tid][poi_records[j]["shop_type"]] += 1
            if poi_records[j]["has_brand"]:
                tract_brand_count[tid] += 1

# Build result
records = []
for tid in tract_ids:
    shop_types = tract_shop_types[tid]
    total_shops = sum(shop_types.values())
    records.append({
        "tract_id": tid,
        "shop_count": total_shops,
        "shop_density_km2": round(total_shops / AREA_KM2, 2),
        "shop_type_entropy": round(shannon_entropy(shop_types), 4),
        "brand_ratio": round(tract_brand_count[tid] / total_shops, 4) if total_shops > 0 else 0.0,
    })

df_shops = pd.DataFrame(records)
print(f"\nCompleted: {len(df_shops)} tracts")
print(f"Mean shops per tract: {df_shops['shop_count'].mean():.1f}")
print(f"Tracts with zero shops: {(df_shops['shop_count'] == 0).sum()}")

In [ ]:
# ── Save output ───────────────────────────────────────
output_path = "csv/09_commercial_density.csv"
df_shops.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_shops)} rows x {df_shops.shape[1]} cols)")
print(df_shops.describe().round(2).to_string())